# SAGE Pipeline: Synthetic QA Generation with Search Agent Evaluation

This notebook demonstrates the SAGE (Search Agent-Guided Evaluation) pipeline for generating
high-quality synthetic QA pairs. Unlike the simpler single/multi-hop approaches, SAGE uses a
multi-stage protocol:

1. **Document Chunking** — Split documents into retrievable chunks
2. **Corpus Indexing** — Upload chunks for BM25 search
3. **SAGE QA Generation** — Generate QA pairs using anchor-based chunk selection and LLM rollouts
4. **Search Agent Filtering** — Validate QA quality via search agent rollouts + LLM judge
5. **Feedback Regeneration** — Refine failed/too-easy items iteratively
6. **Training** — Upload dataset and launch RL training job

The pipeline is orchestrated via the `Pipeline` class which composes four protocol implementations:
- `SageGenerator` — anchor-based or rollout-based question generation
- `SageFilter` — best-of-N search agent rollouts + LLM-as-a-Judge
- `SageFeedbackRegenerator` — refine incorrect/too-easy items via execution feedback
- `SageFormatter` — JSONL output + human-readable report

## 0. Setup

In [12]:
# This is only used internally for development

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
import logging

logging.basicConfig(level=logging.INFO, format="%(name)s — %(levelname)s — %(message)s")

## 1. Document Chunking

RAG systems retrieve chunks, not entire documents. Split your documents into pieces that
can be indexed and retrieved.

In [14]:
from synthetic_data_prep.chunkers.inspector import ChunkInspector
from synthetic_data_prep.chunkers.markdown import MarkdownChunker

# ===== MODIFY HERE: Replace the path and tweak chunking configs =====
DOCS_PATH = "../samples/posthog"

MIN_CHARS = 1024
MAX_CHARS = 2048
OVERLAP_CHARS = 128
# ===== END OF MODIFY SECTION =====

chunker = MarkdownChunker(min_char=MIN_CHARS, max_char=MAX_CHARS, chunk_overlap=OVERLAP_CHARS)
collection = chunker.chunk_folder(DOCS_PATH, file_extensions=[".md", ".mdx"])

inspector = ChunkInspector(collection)
inspector.summary(max_depth=3, max_files_per_folder=4)

ChunkCollection Summary
Total chunks: 3225
Total files: 1183

Chunk Size Statistics:
  Min: 29 chars (contribute/badge.md, chunk 0)
  Max: 2048 chars (data-warehouse/query.mdx, chunk 0)
  Mean: 1148 chars

File Structure:
├── data-warehouse/
│   ├── sql/
│   │   ├── index.mdx (8 chunks)
│   │   ├── useful-functions.mdx (7 chunks)
│   │   ├── variables.mdx (2 chunks)
│   │   └── data-access.mdx (3 chunks)
│   ├── views/
│   │   ├── index.mdx (2 chunks)
│   │   └── materialize.mdx (2 chunks)
│   ├── _snippets/
│   │   └── query-join-example.mdx (1 chunks)
│   ├── sources/
│   │   ├── stripe.mdx (1 chunks)
│   │   ├── tiktok-ads.mdx (1 chunks)
│   │   ├── azure-blob.mdx (1 chunks)
│   │   ├── attio.mdx (1 chunks)
│   │       ... 31 more files
│   ├── under-the-hood.md (1 chunks)
│   ├── query.mdx (4 chunks)
│   ├── tutorials.mdx (1 chunks)
│   ├── changelog.mdx (1 chunks)
│       ... 5 more files
├── how-posthog-works/
│   ├── clickhouse.md (3 chunks)
│   ├── index.mdx (2 chunks)
│   ├── 

## 2. Upload Corpus

Upload your chunks to the Corpora API for BM25 search. This is the retrieval backend
used by both the question generator and the search agent during evaluation.

In [15]:
from synthetic_data_prep.corpus.corpora.source import CorporaChunkSource

# ===== MODIFY HERE: Configure your API and corpus settings =====
API_KEY = "sk_RZEmogqiKirhhDCtipwhSKeqFqnuLUjoxfTQqTkRvNWyuMFOoJkEJxWgJnwoobKS"  # get from https://app.cgft.io/account/api-keys
CORPUS_NAME = "posthog"
BASE_URL = "https://app.cgft.io"
LLM_ENDPOINT = f"{BASE_URL}/api/llm"
LLM_API_KEY = API_KEY
# ===== END OF MODIFY SECTION =====

source = CorporaChunkSource(api_key=API_KEY, corpus_name=CORPUS_NAME, base_url=BASE_URL)

In [16]:
source.populate_from_folder(DOCS_PATH)
collection = source.collection

print(f"Corpus ID: {source.corpus_id}")
print(f"Total chunks: {len(collection)}")

Chunking documents from ../samples/posthog...
ChunkCollection Summary
Total chunks: 3225
Total files: 1183

Chunk Size Statistics:
  Min: 29 chars (contribute/badge.md, chunk 0)
  Max: 2048 chars (data-warehouse/query.mdx, chunk 0)
  Mean: 1148 chars

File Structure:
├── data-warehouse/
│   ├── sql/
│   │   ├── index.mdx (8 chunks)
│   │   ├── useful-functions.mdx (7 chunks)
│   │   ├── variables.mdx (2 chunks)
│   │   └── data-access.mdx (3 chunks)
│   ├── views/
│   │   ├── index.mdx (2 chunks)
│   │   └── materialize.mdx (2 chunks)
│   ├── _snippets/
│   │   └── query-join-example.mdx (1 chunks)
│   ├── sources/
│   │   ├── stripe.mdx (1 chunks)
│   │   ├── tiktok-ads.mdx (1 chunks)
│   │   ├── azure-blob.mdx (1 chunks)
│   │   ├── attio.mdx (1 chunks)
│   │       ... 31 more files
│   ├── under-the-hood.md (1 chunks)
│   ├── query.mdx (4 chunks)
│   ├── tutorials.mdx (1 chunks)
│   ├── changelog.mdx (1 chunks)
│       ... 5 more files
├── how-posthog-works/
│   ├── clickhouse.md (3

httpx — INFO — HTTP Request: GET https://app.cgft.io/api/corpora "HTTP/1.1 200 OK"



Using corpus: posthog (ID: 2e63166e-8ace-42e4-b2c9-1f9a92e96c99)
Uploading 3225 chunks to corpus...


Uploading chunks:   0%|          | 0/33 [00:00<?, ?batch/s]

httpx — INFO — HTTP Request: POST https://app.cgft.io/api/corpora/2e63166e-8ace-42e4-b2c9-1f9a92e96c99/chunks "HTTP/1.1 200 OK"
httpx — INFO — HTTP Request: POST https://app.cgft.io/api/corpora/2e63166e-8ace-42e4-b2c9-1f9a92e96c99/chunks "HTTP/1.1 200 OK"
httpx — INFO — HTTP Request: POST https://app.cgft.io/api/corpora/2e63166e-8ace-42e4-b2c9-1f9a92e96c99/chunks "HTTP/1.1 200 OK"
httpx — INFO — HTTP Request: POST https://app.cgft.io/api/corpora/2e63166e-8ace-42e4-b2c9-1f9a92e96c99/chunks "HTTP/1.1 200 OK"
httpx — INFO — HTTP Request: POST https://app.cgft.io/api/corpora/2e63166e-8ace-42e4-b2c9-1f9a92e96c99/chunks "HTTP/1.1 200 OK"
httpx — INFO — HTTP Request: POST https://app.cgft.io/api/corpora/2e63166e-8ace-42e4-b2c9-1f9a92e96c99/chunks "HTTP/1.1 200 OK"
httpx — INFO — HTTP Request: POST https://app.cgft.io/api/corpora/2e63166e-8ace-42e4-b2c9-1f9a92e96c99/chunks "HTTP/1.1 200 OK"
httpx — INFO — HTTP Request: POST https://app.cgft.io/api/corpora/2e63166e-8ace-42e4-b2c9-1f9a92e96c99/c


Upload complete! Inserted: 3225
Corpus ID: 2e63166e-8ace-42e4-b2c9-1f9a92e96c99
Total chunks: 3225


In [17]:
# Test BM25 search
results = source._client.search(corpus_id=source.corpus_id, query="How do I set up feature flags?", limit=3)
for i, r in enumerate(results):
    print(f"\n--- Result {i + 1} ---")
    print(str(r.content)[:300])

httpx — INFO — HTTP Request: POST https://app.cgft.io/api/corpora/2e63166e-8ace-42e4-b2c9-1f9a92e96c99/search "HTTP/1.1 200 OK"



--- Result 1 ---
## Advanced feature flag patterns  
- [How to use evaluation runtimes and tags together for fine-grained flag control](/tutorials/evaluation-runtimes-and-tags) - Combine runtime and tag filtering for precise flag management
- [How to use feature flag dependencies](/docs/feature-flags/dependencies) -

--- Result 2 ---
### How do I create a flag that targets users who completed an event?  
Although you can't use behavioral cohorts, you still might want to target users who completed an event.  
One option to do this is to create a dynamic behavioral cohort and then duplicate it as a static cohort (using the button 

--- Result 3 ---
---
title: Tutorials and guides
sidebar: Docs
showTitle: true
---
Got a question which isn't answered below? Head to [the community forum](/questions) to let us know!  
- [How to target flags with groups, pages, machines, and more](/docs/feature-flags/targeting-groups)
- [How to set up one-time feat


## 3. Configure the SAGE Pipeline

The SAGE pipeline is configured via `SagePipelineConfig`. You can either:
- Build the config programmatically (shown here)
- Load from a YAML file using `load_sage_config("config.yaml")`

Key configuration areas:
- **Models**: Which LLMs to use for generation, search, judging, and feedback
- **Anchor selection**: How to select and combine chunks for multi-hop questions
- **Refinement**: How many rounds of feedback to apply
- **Pipeline parameters**: Number of samples, rollouts, search steps, etc.

In [18]:
from synthetic_data_prep.qa_generation.sage_utils import (
    AnchorConfig,
    CorpusContextConfig,
    RefinementConfig,
    SageModelConfig,
    SagePipelineConfig,
    SageQueryRewriteConfig,
)

# ===== MODIFY HERE: Configure the SAGE pipeline =====

cfg = SagePipelineConfig(
    # Platform credentials
    api_key=API_KEY,
    base_url=BASE_URL,

    # Corpus
    corpus_id=source.corpus_id,

    # Corpus context — helps the generator create relevant, user-facing questions
    corpus_context=CorpusContextConfig(
        description="PostHog product analytics documentation",
        example_queries=[
            "How do I set up feature flags in React?",
            "What is the difference between events and actions?",
            "How do I configure a reverse proxy for PostHog?",
        ],
    ),

    # Per-agent model configs
    question_generator=SageModelConfig(model="gpt-5.2"),
    search_agent=SageModelConfig(model="gpt-5-mini"),
    judge=SageModelConfig(model="gpt-5.2-chat"),
    feedback=SageModelConfig(model="gpt-5-mini"),

    # Anchor selection — enables intelligent multi-hop chunk pairing
    anchor=AnchorConfig(
        enabled=True,
        bm25_enrichment_top_k=5,
        bm25_enrichment_queries=3,
    ),

    # Refinement controls
    refinement=RefinementConfig(
        max_question_refinements=3,
        max_anchor_regenerations=2,
        hop_tolerance=0,
        min_golden_chunk_overlap=0.5,
    ),

    # Pipeline parameters
    num_samples=5,          # number of seed chunks to sample
    n_search_steps=2,       # target search steps per question
    n_rollouts=4,           # best-of-N rollouts for evaluation
    search_max_turns=16,    # max conversation turns for search agent
    search_max_tool_calls=24,
    max_feedback_rounds=3,  # max filter-regenerate cycles
    min_search_steps=0,     # minimum search steps to include in output
    min_chunk_chars=400,    # minimum chunk size for sampling

    # Output
    output="outputs/sage_output.jsonl",
)

# Fill in agent API keys from the platform key
cfg.resolve_api_keys()

# ===== END OF MODIFY SECTION =====

print(f"Corpus ID: {cfg.corpus_id}")
print(f"Num samples: {cfg.num_samples}")
print(f"N rollouts: {cfg.n_rollouts}")
print(f"Anchor enabled: {cfg.anchor.enabled}")

Corpus ID: 2e63166e-8ace-42e4-b2c9-1f9a92e96c99
Num samples: 5
N rollouts: 4
Anchor enabled: True


### 3.1 Alternative: Load config from YAML

You can also load the full config from a YAML file. See the `SagePipelineConfig` docstring
for the expected format.

In [19]:
# from synthetic_data_prep.qa_generation.sage_utils import load_sage_config
# cfg = load_sage_config("path/to/sage_config.yaml")

## 4. Assemble and Run the SAGE Pipeline

The pipeline follows the `Generator -> Filter -> Regenerator -> Formatter` protocol.
We use iterative mode (`max_rounds > 0`) so that items can be refined multiple times
through the filter-regenerate loop.

In [20]:
from openai import OpenAI

from synthetic_data_prep.qa_generation import (
    GenerationRetryRegenerator,
    Pipeline,
    SageFormatter,
    SageGenerator,
)
from synthetic_data_prep.qa_generation.filters import RetrievalDifficultyFilter
from synthetic_data_prep.trainer.client import RolloutClient

# Create shared clients
rollout_client = RolloutClient(api_key=API_KEY)
llm_client = OpenAI(base_url=f"{LLM_ENDPOINT}", api_key=LLM_API_KEY)

# Assemble the SAGE pipeline
sage_generator = SageGenerator(
    source=source,
    rollout_client=rollout_client,
    qgen_client=llm_client,
    cfg=cfg,
)

pipeline = Pipeline(
    generator=sage_generator,
    filter=RetrievalDifficultyFilter(
        chunk_source=source,
        judge_client=llm_client,
        judge_model=cfg.judge.model,
        top_k=5,
        overlap_threshold=0.5,
        max_refinements=cfg.refinement.max_question_refinements,
    ),
    regenerator=GenerationRetryRegenerator(generator=sage_generator),
    formatter=SageFormatter(cfg=cfg),
    max_rounds=cfg.max_feedback_rounds,  # iterative mode
)

print("Pipeline assembled. Ready to run.")

Pipeline assembled. Ready to run.


In [21]:
# Run the full pipeline
# This will:
# 1. Sample chunks and generate QA pairs (with anchor-based chunk selection)
# 2. Evaluate each QA pair via search agent rollouts + LLM judge
# 3. Refine failed/too-easy items via execution feedback (up to max_rounds)
# 4. Format passing items into JSONL output + report

output = pipeline.run(context={})

synthetic_data_prep.qa_generation.generators.sage — INFO — Corpus capabilities: CorpusCapabilities(doc_ids=True, headers=False, sequential=False) → available types: ['lookup', 'synthesis', 'co_located_multi_hop', 'cross_document_multi_hop']
httpx — INFO — HTTP Request: POST https://app.cgft.io/api/corpora/2e63166e-8ace-42e4-b2c9-1f9a92e96c99/search "HTTP/1.1 200 OK"
httpx — INFO — HTTP Request: POST https://app.cgft.io/api/corpora/2e63166e-8ace-42e4-b2c9-1f9a92e96c99/search "HTTP/1.1 200 OK"
synthetic_data_prep.qa_generation.generators.sage — INFO — Chunk 1: type=lookup, hops=1, refs=3
httpx — INFO — HTTP Request: POST https://app.cgft.io/api/corpora/2e63166e-8ace-42e4-b2c9-1f9a92e96c99/search "HTTP/1.1 200 OK"
httpx — INFO — HTTP Request: POST https://app.cgft.io/api/corpora/2e63166e-8ace-42e4-b2c9-1f9a92e96c99/search "HTTP/1.1 200 OK"
httpx — INFO — HTTP Request: POST https://app.cgft.io/api/corpora/2e63166e-8ace-42e4-b2c9-1f9a92e96c99/search "HTTP/1.1 200 OK"
synthetic_data_prep.qa_

In [22]:
output

{'results': [{'question': 'I’m building a PostHog CDP destination step using Hog plus a follow-up SQL check. An HTTP step returns JSON like `{ "results": [ { "id": "abc123" } ] }`.  \n\n1) What exact **result path** should I use to extract the first result’s `id` into a variable?  \n2) If I then create a Hog **tuple** like `(extractedId, properties[\'$browser\'])`, how do I print the **second** item from that tuple (what index do I use)?  \n3) In a PostHog SQL query, how do I correctly filter events where the `$browser` property equals Chrome, using the right quoting rules for identifiers vs string literals?',
   'answer': "1) Result path: `body.results[0].id`  \n\n2) Tuples in Hog are **1-indexed**, so the second item is accessed with `.2` or `[2]`, e.g.:\n```rust\nprint(myTuple.2)\n// or\nprint(myTuple[2])\n```\n\n3) In PostHog SQL, use **backticks/double quotes for identifiers** and **single quotes for string literals**, e.g.:\n```sql\nSELECT *\nFROM events\nWHERE properties.`$brows

## 5. Inspect Results

The formatter returns a dict with:
- `results` — the final filtered QA pairs
- `all_results` — all passing QA pairs (before min_search_steps filter)
- `stats` — pipeline statistics
- `output_paths` — paths to the JSONL and report files

In [23]:
import json

results = output["results"]
stats = output["stats"]

print(f"Total generated: {stats['total']}")
print(f"Passed: {stats['passed']}")
print(f"Strict passed: {stats['strict_passed']}")
print(f"Dropped by min search steps: {stats['dropped_by_min_step']}")
print(f"Final output: {len(results)} QA pairs")
print(f"\nOutput files:")
for name, path in output["output_paths"].items():
    print(f"  {name}: {path}")

Total generated: 4
Passed: 4
Strict passed: 4
Dropped by min search steps: 0
Final output: 4 QA pairs

Output files:
  jsonl: outputs/sage_output.jsonl
  report: outputs/sage_output.txt


In [27]:
# Inspect rollout metrics
rollout_metrics = stats.get("rollout_metrics", {})
print("Rollout Metrics:")
for key, value in rollout_metrics.items():
    print(f"  {key}: {value}")

# Per-type metrics
print("\nMetrics by QA Type:")
for qa_type, metrics in stats.get("rollout_metrics_by_type", {}).items():
    print(f"  {qa_type}: {metrics}")

# Anchor quality
print("\nAnchor Quality:")
for qa_type, quality in stats.get("anchor_quality", {}).items():
    print(f"  {qa_type}: {quality}")

Rollout Metrics:
  rollouts: 0
  judge_correct: 0
  judge_incorrect: 0
  contract_pass: 0
  contract_fail: 0
  strict_answers: 0
  repaired_answers: 0
  fallback_answers: 0
  repair_attempts: 0
  repair_success: 0
  no_answer_after_retries: 0
  retries_used: 0
  contract_fail_reasons: Counter()

Metrics by QA Type:

Anchor Quality:


In [24]:
# Show sample QA pairs
for i, r in enumerate(results[:3]):
    print(f"\n{'='*80}")
    print(f"QA Pair {i + 1}")
    print(f"{'='*80}")
    print(f"Type: {r['target_qa_type']}")
    print(f"Search steps: {r['search_steps']} (target: {r['target_hop_count']})")
    print(f"Status: {r['status']} ({r['status_reason']})")
    print(f"Refinements: Q={r['question_refinements']}, Anchor={r['anchor_regenerations']}")
    print(f"\nQ: {r['question']}")
    print(f"\nA: {r['answer']}")
    if r.get('golden_chunks'):
        print(f"\nGolden chunks: {len(r['golden_chunks'])}")


QA Pair 1
Type: cross_document_multi_hop
Search steps: 0 (target: 3)
Status: pass (retrieval_difficulty_passed)
Refinements: Q=0, Anchor=0

Q: I’m building a PostHog CDP destination step using Hog plus a follow-up SQL check. An HTTP step returns JSON like `{ "results": [ { "id": "abc123" } ] }`.  

1) What exact **result path** should I use to extract the first result’s `id` into a variable?  
2) If I then create a Hog **tuple** like `(extractedId, properties['$browser'])`, how do I print the **second** item from that tuple (what index do I use)?  
3) In a PostHog SQL query, how do I correctly filter events where the `$browser` property equals Chrome, using the right quoting rules for identifiers vs string literals?

A: 1) Result path: `body.results[0].id`  

2) Tuples in Hog are **1-indexed**, so the second item is accessed with `.2` or `[2]`, e.g.:
```rust
print(myTuple.2)
// or
print(myTuple[2])
```

3) In PostHog SQL, use **backticks/double quotes for identifiers** and **single qu

## 6. Upload Dataset and Launch Training

Upload the generated SAGE dataset and launch an RL training job.

In [ ]:
from synthetic_data_prep.trainer.pipeline import train
from synthetic_data_prep.envs import CorporaSearchEnv

# Prepare train/eval split from SAGE results
import random

random.seed(42)
qa_pairs = [{"question": r["question"], "answer": r["answer"]} for r in results]
random.shuffle(qa_pairs)

split_idx = max(1, int(len(qa_pairs) * 0.8))
train_data = qa_pairs[:split_idx]
eval_data = qa_pairs[split_idx:]

print(f"Train: {len(train_data)}, Eval: {len(eval_data)}")

In [ ]:
# ===== MODIFY HERE: Configure training =====
EXPERIMENT_NAME = "sage-posthog"
# ===== END OF MODIFY SECTION =====

experiment_id = train(
    env_class=CorporaSearchEnv,
    env_args={
        "api_key": API_KEY,
        "corpus_id": source.corpus_id,
        "base_url": BASE_URL,
    },
    train_dataset=train_data,
    eval_dataset=eval_data,
    prefix="sage-search",
    api_key=API_KEY,
    base_url=BASE_URL,
    experiment_name=EXPERIMENT_NAME,
)

print(f"Experiment ID: {experiment_id}")